# Expected checks (04_02 section 13)

Written from the Week 3 rules (`AGENTS.md`, 03_02 section 6) and the 04_02 guidance. Each row says what a correct result must look like **before** reading the audit output. The last column maps it to the audit rows in `outputs/week4_validation/validation_table.csv` that record the observed result.

> **Honesty note:** Claude drafted this table on 2026-09-27, after the audit had already run. It was not written before the results. Say so in the AI-use statement.

| # | Check | Expected result / tolerance | Evidence | What failure would mean | Audit rows |
|---|---|---|---|---|---|
| 1 | Exact endpoints | Every `ok` route starts at the easterly `route1_short` endpoint and ends at the westerly one, within **0.01 m** | `routes.gpkg`, `route1_short.gpkg` | The agent solved a different routing problem, or snapped the endpoints wrongly | `R-*-ends` |
| 2 | Continuity and validity | Each route is one non-empty, valid `LineString` with no gaps or self-loops | `routes.gpkg` | The route is fragmented or malformed, so its length and cost mean nothing | `R-*-valid`, `R-*-simple` |
| 3 | Study-area containment | All route and connector geometry lies inside the baseline bounds plus **3,000 m** | `routes.gpkg`, `connectors.gpkg`, rebuilt rectangle | The agent used data outside its allowed environment | `R-*-contain`, `R-*-conn*` |
| 4 | Lengths in metres | Measured in EPSG:32737; geometry length equals the traversed-edge sum within **0.01 m**; UTM vs geodesic length within **0.5 %**; no route shorter than the straight-line distance between its endpoints (a network route may be longer than the baseline) | `routes.gpkg`, `route_comparison.csv` | CRS/degree error, double-counted edges, or impossible geometry | `R-*-len`, `R-*-units`, `R-*-straight` |
| 5 | Connector limits | Every gap connector is ≤ the run's gap tolerance (**50 m** original, **25 m** rerun); every endpoint connector is ≤ **500 m**; each connector is counted **once** in `connector_m`; the baseline has 0 connectors | `connectors.gpkg`, `route_comparison.csv` | Invented infrastructure beyond the rules, or connector length double-counted | `R-*-conn*`, `C-base` |
| 6 | Cost arithmetic | `estimated_cost_usd = length_m × 10` within **USD 0.01**, using one common rate for every candidate | `route_comparison.csv` | Arithmetic error; ranking by cost would be wrong | `R-*-cost`, `C-rate`, `S-rate` |
| 7 | Unavailable routes | All four rows present; a failed candidate has a named status (`no_path`, `endpoint_too_far`, `empty_network`) and a **blank** length/cost (not 0), and no geometry in `routes.gpkg` | `route_comparison.csv`, `routes.gpkg` | A failure silently dropped or disguised as a free or zero-length route | `C-rows`, `C-unavail`, `C-gpkg`, `U-power` |
| 8 | Synthetic known answers | The five `AGENTS.md` cases give their known outcome (e.g. disconnected endpoints give `no_path`); the hand case 80 m + 120 m with a 30 m gap gives **230 m, USD 2,300**, with the connector counted once | `routing_report.md` test table, audit oracle | A logic bug that real data would hide | `T-*`, `T-oracle*` |
| 9 | Controlled sensitivity | Rerunning at a **25 m** gap tolerance changes only that one setting; any change in status, length, connectors or ranking is reported, not tuned away | original and `sensitivity_gap_25m` reports | The comparison isn't controlled, or a threshold was bent to force a result | `S-*` |
| 10 | Transfers at real junctions | A route switches between OSM features only at a vertex both features share; any other transfer is flagged | `routes.gpkg`, source layers | The route may cross a bridge or tunnel as if it were an at-grade junction | `V-crossings` |

**Claim geometry cannot establish:** that any route can actually be built. A 2D crossing may hide a bridge or tunnel, and running near a road, rail line or power line gives no access rights, permits or construction feasibility. Whether a power line exists near the start point also can't be settled from OSM alone. These need imagery, utility records or legal and engineering evidence (the audit's `E-*` rows).
